In [3]:
import subprocess
subprocess.run(["pip", "install", "openpyxl", "pandas",
                "pdfplumber", "python-docx"], capture_output=True)

import warnings, openpyxl, pandas as pd
import pdfplumber, re, json, pathlib, datetime, os
from docx import Document
warnings.filterwarnings("ignore")

print("✓ openpyxl:", openpyxl.__version__)
print("✓ pandas:", pd.__version__)
print("✓ pdfplumber, python-docx: OK")
print("✓ All imports successful")

✓ openpyxl: 3.1.2
✓ pandas: 2.2.2
✓ pdfplumber, python-docx: OK
✓ All imports successful


In [5]:
import os, pathlib, shutil, warnings, openpyxl, pdfplumber, re
warnings.filterwarnings("ignore")

# ── Folders ───────────────────────────────────────────────────────────
INPUT_DIR  = pathlib.Path("input")
OUTPUT_DIR = pathlib.Path("output")
INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Source: your local Downloads folder ──────────────────────────────
SOURCE_DIR = pathlib.Path(r"C:\Users\hp\Downloads\For Students (1)\For Students\inputs")

EXPECTED = [
    "1. IVC DOE R2 (Input).xlsx",
    "2. City of York Council (Input).xlsx",
    "3. Digital Security IT Sample Register (Input).xlsx",
    "4. Moorgate Crossrail Register (Input).xlsx",
    "5. Corporate_Risk_Register (Input).pdf",
]

# ── Copy files from Downloads into input/ ────────────────────────────
print("Copying files from Downloads...\n")
for fname in EXPECTED:
    src  = SOURCE_DIR / fname
    dest = INPUT_DIR  / fname
    if dest.exists():
        print(f"  ⏭  already exists, skipping  → {fname}")
    elif src.exists():
        shutil.copy(src, dest)
        print(f"  ✓  copied → input/{fname}")
    else:
        print(f"  ✗  NOT FOUND in source: {src}")

# ── Verify all 5 files landed in input/ ──────────────────────────────
print("\nVerifying...\n")
all_found = True
for fname in EXPECTED:
    path   = INPUT_DIR / fname
    status = "✓ FOUND  " if path.exists() else "✗ MISSING"
    size   = f"({path.stat().st_size // 1024} KB)" if path.exists() else ""
    print(f"  {status}  {fname}  {size}")
    if not path.exists(): all_found = False

if not all_found:
    raise FileNotFoundError(
        "One or more files are missing.\n"
        "Check that SOURCE_DIR points to the correct folder and re-run."
    )

# ── Inspect each file's raw structure ────────────────────────────────
print("\n── Inspecting file structures ──\n")
for f in sorted(INPUT_DIR.iterdir()):
    print(f"\n{'='*55}\nFILE: {f.name}\n{'='*55}")
    if f.suffix == ".xlsx":
        wb = openpyxl.load_workbook(f, data_only=True)
        print("Sheets:", wb.sheetnames)
        for sheet in wb.sheetnames:
            ws = wb[sheet]
            print(f"\n  Sheet: '{sheet}' | Dims: {ws.dimensions}")
            for i, row in enumerate(ws.iter_rows(values_only=True)):
                if i < 3 and any(v is not None for v in row):
                    print(f"    Row {i+1}:", [str(v)[:40] if v else "" for v in row])
    elif f.suffix == ".pdf":
        with pdfplumber.open(f) as pdf:
            print(f"Pages: {len(pdf.pages)}")
            tables = pdf.pages[0].extract_tables()
            if tables:
                print(f"Tables on page 1: {len(tables)}")
                for row in tables[0][:4]:
                    cleaned = [re.sub(r"\s+", " ", str(c)).strip()[:40]
                               if c else "" for c in row]
                    print("  ", cleaned)
            else:
                print("  Text:", pdf.pages[0].extract_text()[:300])

Copying files from Downloads...

  ✓  copied → input/1. IVC DOE R2 (Input).xlsx
  ✓  copied → input/2. City of York Council (Input).xlsx
  ✓  copied → input/3. Digital Security IT Sample Register (Input).xlsx
  ✓  copied → input/4. Moorgate Crossrail Register (Input).xlsx
  ✓  copied → input/5. Corporate_Risk_Register (Input).pdf

Verifying...

  ✓ FOUND    1. IVC DOE R2 (Input).xlsx  (112 KB)
  ✓ FOUND    2. City of York Council (Input).xlsx  (21 KB)
  ✓ FOUND    3. Digital Security IT Sample Register (Input).xlsx  (16 KB)
  ✓ FOUND    4. Moorgate Crossrail Register (Input).xlsx  (17 KB)
  ✓ FOUND    5. Corporate_Risk_Register (Input).pdf  (686 KB)

── Inspecting file structures ──


FILE: 1. IVC DOE R2 (Input).xlsx
Sheets: ['Intro', 'Risk Register', 'Valid data fields']

  Sheet: 'Intro' | Dims: B1:L44
    Row 1: ['', 'Note: Mock data currently within risk re', '', '', '', '', '', '', '', '', '', '']
    Row 3: ['', 'MHK risk register template, version: Sep', '', '', '', '', '', '', 

In [7]:
import pathlib, openpyxl, warnings
warnings.filterwarnings("ignore")

# ── Point this at your corrected outputs folder ───────────────────────
OUTPUT_CORRECT = pathlib.Path(
    r"C:\Users\hp\Downloads\For Students (1)\For Students\outputs"
)

# ── Inspect each corrected file ───────────────────────────────────────
for f in sorted(OUTPUT_CORRECT.glob("*.xlsx")):
    print(f"\n{'='*55}\n{f.name}\n{'='*55}")
    wb = openpyxl.load_workbook(f)
    print("Sheets:", wb.sheetnames)

    ws = wb['Simplified Register']
    headers = [c.value for c in ws[1] if c.value]
    print("\nHeaders:", headers)
    for row in ws.iter_rows(min_row=2, max_row=4, values_only=True):
        if any(v is not None for v in row):
            print("  ", [str(v)[:35] if v else "" for v in row])

    ws2 = wb['Output Requirements']
    print("\nOutput Requirements:")
    for row in ws2.iter_rows(values_only=True):
        if row[0]: print(f"  {row[0]}")

# ── Unified mandatory columns ─────────────────────────────────────────
print("\n── Unified Mandatory Columns ──")
for col in ["Risk ID", "Risk Description", "Project Stage",
            "Project Category", "Risk Owner", "Mitigating Action",
            "Likelihood (1-10)", "Impact (1-10)",
            "Risk Priority (low, med, high)"]:
    print(f"  ✓ {col}")


1. IVC DOE (Final).xlsx
Sheets: ['Simplified Register', 'Output Requirements']

Headers: ['Date Added', 'Risk ID', 'Risk Description', 'Project Stage', 'Project Category', 'Risk Owner', 'Likelihood (1-10) (pre-mitigation)', 'Impact (1-10) (pre-mitigation)', 'Risk Priority (pre-mitigation)', 'Mitigating Action', 'Likelihood (1-10) (post-mitigation)', 'Impact (1-10) (post-mitigation)', 'Risk Priority (post-mitigation)']
   ['A', '', 'N', 'G', '', 'H', 'L', 'K', 'M', 'P', 'L', 'K', 'M']
   ['2017-02-21 00:00:00', '1', 'Data sharing from BBSRI at risk', 'Pre-construction', 'Planning', 'Environmental', '8', '4', 'Med', 'facilitate conversations with Umain', '6', '2', 'Low']
   ['2017-02-21 00:00:00', '2', 'Ice data collection unsuccessful', 'Construction', 'Services', 'UAA lead', '6', '6', 'Low', 'real time data collection implement', '2', '4', 'Low']

Output Requirements:
  The following columns are mandatory for the final risk registers. Additional columns may be added, if information is

In [9]:
def parse_xlsx(filepath):
    wb = openpyxl.load_workbook(filepath, data_only=True)
    parts = []
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        parts.append(f"\n--- SHEET: {sheet_name} ---")
        for row in ws.iter_rows(values_only=True):
            cells = [str(c).strip() if c is not None else "" for c in row]
            line = " | ".join(cells).strip(" |")
            if line.replace("|", "").strip(): parts.append(line)
    return "\n".join(parts)

def parse_pdf(filepath):
    parts = []
    with pdfplumber.open(filepath) as pdf:
        for pg, page in enumerate(pdf.pages, 1):
            parts.append(f"\n--- PAGE {pg} ---")
            tables = page.extract_tables()
            if tables:
                for table in tables:
                    for row in table:
                        if row:
                            cleaned = [re.sub(r"\s+", " ", str(c)).strip()
                                       if c else "" for c in row]
                            line = " | ".join(cleaned).strip(" |")
                            if line.replace("|", "").strip(): parts.append(line)
            else:
                text = page.extract_text()
                if text: parts.append(text)
    return "\n".join(parts)

def parse_docx(filepath):
    doc = Document(str(filepath))
    parts = []
    for para in doc.paragraphs:
        if para.text.strip(): parts.append(para.text.strip())
    for table in doc.tables:
        parts.append("--- TABLE ---")
        for row in table.rows:
            parts.append(" | ".join(cell.text.strip() for cell in row.cells))
    return "\n".join(parts)

# ── Test all parsers ──────────────────────────────────────────────────
for f in sorted(INPUT_DIR.iterdir()):
    if f.suffix == ".xlsx":  text = parse_xlsx(f)
    elif f.suffix == ".pdf": text = parse_pdf(f)
    else: continue
    lines = [l for l in text.split("\n") if l.strip()]
    print(f"✓ {f.name}: {len(lines)} lines extracted")

✓ 1. IVC DOE R2 (Input).xlsx: 1022 lines extracted
✓ 2. City of York Council (Input).xlsx: 77 lines extracted
✓ 3. Digital Security IT Sample Register (Input).xlsx: 17 lines extracted
✓ 4. Moorgate Crossrail Register (Input).xlsx: 108 lines extracted
✓ 5. Corporate_Risk_Register (Input).pdf: 1716 lines extracted


In [11]:
LIKELIHOOD_MAP = {
    "rare": 1, "very low": 1, "unlikely": 3, "possible": 5,
    "likely": 7, "probable": 7, "almost certain": 9, "certain": 10
}
IMPACT_MAP = {
    "negligible": 1, "minor": 3, "moderate": 5, "serious": 7,
    "major": 8, "critical": 9, "catastrophic": 10, "extreme": 10
}

def to_num(v):
    """Convert any likelihood/impact value to float 1-10 or None."""
    if v is None: return None
    if isinstance(v, (int, float)): return round(float(v), 2)
    s = str(v).strip().lower()
    try: return round(float(s), 2)
    except: pass
    for k, n in {**LIKELIHOOD_MAP, **IMPACT_MAP}.items():
        if k in s: return float(n)
    return None

def to_pri(v, l=None, i=None):
    """Return 'Low', 'Med', or 'High' from label or likelihood x impact."""
    if v:
        s = str(v).strip().lower()
        if "high" in s: return "High"
        if "med" in s or "medium" in s: return "Med"
        if "low" in s: return "Low"
    if l and i:
        try:
            sc = float(l) * float(i)
            return "Low" if sc <= 20 else ("Med" if sc <= 50 else "High")
        except: pass
    return None

def to_date(v):
    """Return a date object or None from common date formats."""
    if v is None: return None
    if isinstance(v, (datetime.datetime, datetime.date)): return v
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y"):
        try: return datetime.datetime.strptime(str(v).strip(), fmt).date()
        except: pass
    return None

def derive_output_name(input_filepath):
    """Swap (Input) -> (Final), fix underscores, keep .xlsx extension."""
    stem = pathlib.Path(input_filepath).stem
    stem = stem.replace("(Input)", "(Final)").replace("_", " ")
    stem = re.sub(r" {2,}", " ", stem).strip()
    return stem + ".xlsx"

# ── Quick tests ───────────────────────────────────────────────────────
print("to_num tests:")
for v in [7, "5", "Possible", "Minor", "Serious", None]:
    print(f"  {repr(v):20} → {to_num(v)}")
print("\nto_pri tests:")
for v, l, i in [("High", None, None), (None, 5, 3), (None, 8, 8)]:
    print(f"  val={repr(v):8} L={l} I={i} → {to_pri(v, l, i)}")
print("\nderive_output_name tests:")
for name in ["4. Moorgate Crossrail Register (Input).xlsx",
             "5. Corporate_Risk_Register (Input).pdf"]:
    print(f"  {name} → {derive_output_name(name)}")

to_num tests:
  7                    → 7.0
  '5'                  → 5.0
  'Possible'           → 5.0
  'Minor'              → 3.0
  'Serious'            → 7.0
  None                 → None

to_pri tests:
  val='High'   L=None I=None → High
  val=None     L=5 I=3 → Low
  val=None     L=8 I=8 → High

derive_output_name tests:
  4. Moorgate Crossrail Register (Input).xlsx → 4. Moorgate Crossrail Register (Final).xlsx
  5. Corporate_Risk_Register (Input).pdf → 5. Corporate Risk Register (Final).xlsx


In [13]:

# ── File 1: IVC DOE R2 (complex multi-header, pre + post mitigation) ──
wb1 = openpyxl.load_workbook(INPUT_DIR / "1. IVC DOE R2 (Input).xlsx", data_only=True)
risks_1 = []
stage_map = {"pre": "Pre-construction", "construct": "Construction",
             "operat": "Operations", "commission": "Commissioning",
             "design": "Pre-construction"}
for row in list(wb1["Risk Register"].iter_rows(values_only=True))[3:]:
    name = str(row[3]).strip() if row[3] else ""
    if not name or name.lower() in ("none", ""): continue
    def s5(v): return round(float(v) * 2, 1) if v and to_num(v) else None
    lp = s5(to_num(row[11])); ip = s5(to_num(row[10]))
    lq = s5(to_num(row[18])); iq = s5(to_num(row[17]))
    sr = str(row[6]).strip() if row[6] else None
    stage = next((v for k, v in stage_map.items() if sr and k in sr.lower()), sr)
    risks_1.append({
        "date_added": to_date(row[0]), "risk_id": str(len(risks_1) + 1),
        "risk_description": name, "project_stage": stage,
        "project_category": str(row[1]).strip() if row[1] else None,
        "risk_owner": str(row[7]).strip() if row[7] else None,
        "likelihood_pre": lp, "impact_pre": ip,
        "priority_pre": to_pri(None, lp, ip),
        "mitigating_action": str(row[15]).strip() if row[15] else None,
        "likelihood_post": lq, "impact_post": iq,
        "priority_post": to_pri(None, lq, iq), "result": None
    })

# ── File 2: City of York Council ──────────────────────────────────────
wb2 = openpyxl.load_workbook(
    INPUT_DIR / "2. City of York Council (Input).xlsx", data_only=True)
risks_2 = [{"date_added": to_date(r[0]), "risk_id": str(r[1]) if r[1] else None,
    "risk_description": str(r[2]).strip(),
    "project_stage": str(r[4]).strip() if r[4] else None,
    "project_category": str(r[5]).strip() if r[5] else None,
    "risk_owner": str(r[9]).strip() if r[9] else None,
    "likelihood_pre": to_num(r[6]), "impact_pre": to_num(r[7]),
    "priority_pre": to_pri(None, to_num(r[6]), to_num(r[7])),
    "mitigating_action": str(r[3]).strip() if r[3] else None,
    "likelihood_post": None, "impact_post": None, "priority_post": None,
    "result": str(r[10]).strip() if r[10] else None}
    for r in list(wb2["Sheet1"].iter_rows(values_only=True))[1:] if r[2]]

# ── File 3: Digital Security IT ───────────────────────────────────────
wb3 = openpyxl.load_workbook(
    INPUT_DIR / "3. Digital Security IT Sample Register (Input).xlsx", data_only=True)
CAT_KW = {
    "Cybersecurity": ["iam", "identity", "access", "cyber", "authentication"],
    "Infrastructure": ["backup", "recovery", "infrastructure", "database"],
    "Data Management": ["data", "gdpr", "privacy"],
    "Compliance": ["compliance", "iso", "framework"]
}
def ic3(d): return next((c for c, kw in CAT_KW.items()
    if any(k in d.lower() for k in kw)), "IT Risk")
risks_3 = [{"date_added": to_date(r[0]), "risk_id": str(r[1]) if r[1] else None,
    "risk_description": str(r[2]).strip(),
    "project_stage": str(r[3]).strip() if r[3] else "Operations",
    "project_category": str(r[4]).strip() if r[4] else ic3(str(r[2])),
    "risk_owner": str(r[8]).strip() if r[8] else None,
    "likelihood_pre": to_num(r[5]), "impact_pre": to_num(r[6]),
    "priority_pre": to_pri(str(r[7]) if r[7] else None, to_num(r[5]), to_num(r[6])),
    "mitigating_action": str(r[9]).strip() if r[9] else None,
    "likelihood_post": None, "impact_post": None, "priority_post": None, "result": None}
    for r in list(wb3["Sheet1"].iter_rows(values_only=True))[1:] if r[2]]

# ── File 4: Moorgate Crossrail (qualitative labels → numeric) ─────────
wb4 = openpyxl.load_workbook(
    INPUT_DIR / "4. Moorgate Crossrail Register (Input).xlsx", data_only=True)
CR_CAT = {
    "Programme":   ["delay", "programme", "schedule", "slippage"],
    "Contractor":  ["contractor", "riney", "highways", "expire"],
    "Stakeholder": ["crossrail", "tfl", "engagement"],
    "Development": ["development", "developer", "surrounding"],
    "Legal":       ["legal", "compulsory", "cpo"],
    "Utilities":   ["utility", "utilities"]
}
def ic4(d): return next((c for c, kw in CR_CAT.items()
    if any(k in d.lower() for k in kw)), "Project Risk")
risks_4 = [{"date_added": to_date(r[0]), "risk_id": str(r[1]) if r[1] else None,
    "risk_description": str(r[2]).replace("\n", " ").strip() if r[2] else None,
    "project_stage": str(r[3]).strip() if r[3] else "Construction",
    "project_category": str(r[4]).strip() if r[4] else ic4(str(r[2])),
    "risk_owner": str(r[8]).strip() if r[8] else None,
    "likelihood_pre": to_num(r[5]), "impact_pre": to_num(r[6]),
    "priority_pre": to_pri(str(r[7]) if r[7] else None, to_num(r[5]), to_num(r[6])),
    "mitigating_action": str(r[9]).replace("\n", " ").strip() if r[9] else None,
    "likelihood_post": None, "impact_post": None, "priority_post": None, "result": None}
    for r in list(wb4["Sheet1"].iter_rows(values_only=True))[1:] if r[2]]

# ── File 5: Corporate Risk Register PDF (rotated headers) ─────────────
PDF_CATS = {
    "Emergency Planning":     ["disaster", "emergency", "flood", "resilience"],
    "Information Technology": ["it ", "data", "system", "cyber", "gdpr", "digital"],
    "Financial":              ["financial", "fraud", "budget", "cost", "funding"],
    "Legal & Compliance":     ["legal", "compliance", "regulation", "statutory"],
    "Human Resources":        ["staff", "workforce", "recruit", "skill", "people"],
    "Governance":             ["governance", "scrutiny", "council", "democracy"],
    "Health & Safety":        ["health", "safety", "accident", "injury"]
}
def ic5(d): return next((c for c, kw in PDF_CATS.items()
    if any(k in d.lower() for k in kw)), "Corporate Risk")
risks_5 = []; seen = set()
with pdfplumber.open(INPUT_DIR / "5. Corporate_Risk_Register (Input).pdf") as pdf:
    for page in pdf.pages:
        for table in page.extract_tables():
            for row in table:
                if not row or len(row) < 24: continue
                c = [re.sub(r"\s+", " ", str(x)).strip() if x else "" for x in row]
                ref = c[1]
                if not ref or not re.match(r"^\d+$", ref) or ref in seen: continue
                seen.add(ref)
                rt = c[3]
                if not rt or len(rt) < 10: continue
                desc = rt
                if "Risk:-" in rt:
                    pts = rt.split("Effects:-")
                    desc = pts[0].replace("Risk:-", "").strip()
                    if len(pts) > 1: desc += " | Effects: " + pts[1].strip()
                ip = to_num(c[4] or c[5]); lp = to_num(c[7] or c[8])
                iq = to_num(c[15] if len(c) > 15 else "")
                lq = to_num(c[18] if len(c) > 18 else "")
                mit = c[13] if len(c) > 13 else ""
                act = c[24] if len(c) > 24 else ""
                full = ((mit + " | " + act) if mit and act else mit or act).replace("\uf0b7", "•")
                risks_5.append({
                    "date_added": None, "risk_id": ref,
                    "risk_description": desc.replace("\uf0b7", "").strip(),
                    "project_stage": "Operations", "project_category": ic5(desc),
                    "risk_owner": c[23].replace("\n", " ").strip() if len(c) > 23 else None,
                    "likelihood_pre": lp, "impact_pre": ip,
                    "priority_pre": to_pri(None, lp, ip),
                    "mitigating_action": full[:2000] if full else None,
                    "likelihood_post": lq, "impact_post": iq,
                    "priority_post": to_pri(None, lq, iq),
                    "result": c[26] if len(c) > 26 else None
                })

# ── Summary ───────────────────────────────────────────────────────────
for name, risks in [("File 1 (IVC DOE)", risks_1), ("File 2 (York Council)", risks_2),
                    ("File 3 (Digital Security)", risks_3), ("File 4 (Moorgate)", risks_4),
                    ("File 5 (Corporate PDF)", risks_5)]:
    print(f"✓ {name}: {len(risks)} risks extracted")

✓ File 1 (IVC DOE): 33 risks extracted
✓ File 2 (York Council): 45 risks extracted
✓ File 3 (Digital Security): 3 risks extracted
✓ File 4 (Moorgate): 10 risks extracted
✓ File 5 (Corporate PDF): 15 risks extracted


In [15]:
MANDATORY = ["risk_id", "risk_description", "project_stage", "project_category",
             "risk_owner", "mitigating_action", "likelihood_pre",
             "impact_pre", "priority_pre"]

pairs = [
    ("File 1", risks_1, OUTPUT_CORRECT / "1. IVC DOE (Final).xlsx"),
    ("File 2", risks_2, OUTPUT_CORRECT / "2. City of York Council (Final).xlsx"),
    ("File 3", risks_3, OUTPUT_CORRECT / "3. Digital Security IT Sample Register (Final).xlsx"),
]

for label, risks, correct_path in pairs:
    wb_c = openpyxl.load_workbook(correct_path)
    correct_rows = [r for r in wb_c['Simplified Register'].iter_rows(
                        min_row=2, values_only=True) if any(v for v in r)]
    match = ("✓ MATCH" if len(risks) == len(correct_rows)
             else f"⚠ DIFF  (extracted={len(risks)} correct={len(correct_rows)})")
    print(f"\n{label}: row count {match}")
    for field in MANDATORY:
        filled = sum(1 for r in risks if r.get(field) not in (None, "", "None"))
        pct    = round(filled / len(risks) * 100) if risks else 0
        icon   = "✓" if pct >= 80 else ("⚠" if pct >= 40 else "✗")
        print(f"  {icon} {field:35} {filled}/{len(risks)} ({pct}%)")


File 1: row count ✓ MATCH
  ✓ risk_id                             33/33 (100%)
  ✓ risk_description                    33/33 (100%)
  ✓ project_stage                       33/33 (100%)
  ✓ project_category                    33/33 (100%)
  ✓ risk_owner                          33/33 (100%)
  ✓ mitigating_action                   33/33 (100%)
  ✓ likelihood_pre                      33/33 (100%)
  ✓ impact_pre                          33/33 (100%)
  ✓ priority_pre                        33/33 (100%)

File 2: row count ✓ MATCH
  ✓ risk_id                             45/45 (100%)
  ✓ risk_description                    45/45 (100%)
  ✓ project_stage                       45/45 (100%)
  ✓ project_category                    45/45 (100%)
  ✓ risk_owner                          45/45 (100%)
  ✓ mitigating_action                   44/45 (98%)
  ✓ likelihood_pre                      45/45 (100%)
  ✓ impact_pre                          45/45 (100%)
  ✓ priority_pre                        45/45 

In [17]:
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

HEADER_FILL = PatternFill("solid", fgColor="1F4E79")
HEADER_FONT = Font(color="FFFFFF", bold=True)
H_ALIGN     = Alignment(horizontal="center", vertical="center", wrap_text=True)
C_ALIGN     = Alignment(vertical="top", wrap_text=True)
THIN        = Side(border_style="thin", color="B0B0B0")
BORDER      = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
ODD_FILL    = PatternFill("solid", fgColor="DEEAF1")
EVEN_FILL   = PatternFill("solid", fgColor="FFFFFF")

REQ_ROWS = [
    ("The following columns are mandatory for the final risk registers. "
     "Additional columns may be added, if information is provided in the "
     "input files (ex. date, additional comments etc):", None, None),
    ("Risk ID", "If not provided, any identifier may be used.", None),
    ("Risk Description", None, None),
    ("Project Stage", "Required for construction or project based risk registers.", None),
    ("Project Category", None, None), ("Risk Owner", None, None),
    ("Mitigating Action", None, None),
    ("Likelihood (1-10)", "If multiple stages, all should be reflected in the output", None),
    ("Impact (1-10)", None, None),
    ("Risk Priority (low, med, high)", None, None),
]

def write_xlsx(headers, rows, out_path):
    """Write a styled two-sheet workbook to out_path."""
    wb = openpyxl.Workbook()
    ws = wb.active; ws.title = "Simplified Register"
    ws.append(headers)
    for cell in ws[1]:
        cell.font = HEADER_FONT; cell.fill = HEADER_FILL
        cell.alignment = H_ALIGN; cell.border = BORDER
    ws.freeze_panes = "A2"
    for ri, row in enumerate(rows, start=2):
        ws.append(row)
        fill = ODD_FILL if ri % 2 == 0 else EVEN_FILL
        for cell in ws[ri]:
            cell.alignment = C_ALIGN; cell.border = BORDER; cell.fill = fill
    for col in ws.columns:
        maxlen = max((len(str(c.value)) if c.value else 0 for c in col), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(maxlen + 4, 60)
    ws2 = wb.create_sheet("Output Requirements")
    for r in REQ_ROWS: ws2.append(r)
    wb.save(out_path)
    print(f"  ✓ {out_path.name}  ({len(rows)} rows)")

def build_rows(risks, pre_post, has_result=False):
    """Convert risk dicts into (headers, rows) for Excel output."""
    if pre_post:
        headers = ["Date Added", "Risk ID", "Risk Description", "Project Stage",
                   "Project Category", "Risk Owner",
                   "Likelihood (1-10) (pre-mitigation)",
                   "Impact (1-10) (pre-mitigation)",
                   "Risk Priority (pre-mitigation)", "Mitigating Action",
                   "Likelihood (1-10) (post-mitigation)",
                   "Impact (1-10) (post-mitigation)",
                   "Risk Priority (post-mitigation)"]
    else:
        headers = ["Date Added", "Risk ID", "Risk Description", "Project Stage",
                   "Project Category", "Risk Owner", "Likelihood (1-10)",
                   "Impact (1-10)", "Risk Priority (low, med, high)", "Mitigating Action"]
        if has_result: headers.append("Result")
    rows = []
    for r in risks:
        lp = r.get("likelihood_pre");  ip = r.get("impact_pre")
        pp = r.get("priority_pre")  or to_pri(None, lp, ip)
        lq = r.get("likelihood_post"); iq = r.get("impact_post")
        pq = r.get("priority_post") or to_pri(None, lq, iq)
        row = [r.get("date_added"), r.get("risk_id"), r.get("risk_description"),
               r.get("project_stage"), r.get("project_category"), r.get("risk_owner"),
               lp, ip, pp, r.get("mitigating_action")]
        if pre_post:     row += [lq, iq, pq]
        elif has_result: row.append(r.get("result"))
        rows.append(row)
    return headers, rows

# ── Write all 5 output files ──────────────────────────────────────────
OUTPUT_DIR = pathlib.Path("output"); OUTPUT_DIR.mkdir(exist_ok=True)
files = [
    (risks_1, "1. IVC DOE R2 (Final).xlsx",                         True,  False),
    (risks_2, "2. City of York Council (Final).xlsx",                False, True),
    (risks_3, "3. Digital Security IT Sample Register (Final).xlsx", False, False),
    (risks_4, "4. Moorgate Crossrail Register (Final).xlsx",         False, False),
    (risks_5, "5. Corporate Risk Register (Final).xlsx",             True,  True),
]
print("Writing output files...\n")
for risks, fname, pre_post, has_result in files:
    headers, rows = build_rows(risks, pre_post, has_result)
    write_xlsx(headers, rows, OUTPUT_DIR / fname)

Writing output files...

  ✓ 1. IVC DOE R2 (Final).xlsx  (33 rows)
  ✓ 2. City of York Council (Final).xlsx  (45 rows)
  ✓ 3. Digital Security IT Sample Register (Final).xlsx  (3 rows)
  ✓ 4. Moorgate Crossrail Register (Final).xlsx  (10 rows)
  ✓ 5. Corporate Risk Register (Final).xlsx  (15 rows)


In [19]:
MANDATORY_CHECK = ["Risk ID", "Risk Description", "Likelihood",
                   "Impact", "Risk Priority", "Mitigating Action"]
all_pass = True

print("── Final Quality Check ──\n")
for risks, fname, _, _ in files:
    path      = OUTPUT_DIR / fname
    wb        = openpyxl.load_workbook(path)
    sheets    = wb.sheetnames
    ws        = wb["Simplified Register"]
    headers   = [c.value for c in ws[1] if c.value]
    data_rows = [r for r in ws.iter_rows(min_row=2, values_only=True)
                 if any(v for v in r)]
    has_sheets = ("Simplified Register" in sheets
                  and "Output Requirements" in sheets)
    has_data   = len(data_rows) > 0
    man_ok     = all(any(m.lower() in h.lower() for h in headers)
                     for m in MANDATORY_CHECK)
    status = "PASS ✓" if (has_sheets and has_data and man_ok) else "FAIL ✗"
    if "FAIL" in status: all_pass = False
    print(f"  {status}  {fname}  ({len(data_rows)} rows, {len(headers)} cols)")

print()
if all_pass:
    print("✅ All files passed. Ready for submission.")
    print("\nFiles to include in your submission zip:")
    print("  → output/4. Moorgate Crossrail Register (Final).xlsx")
    print("  → output/5. Corporate Risk Register (Final).xlsx")
    print("  → your code folder with model.py")
else:
    print("⚠  Some files failed — review the checks above before submitting.")

── Final Quality Check ──

  PASS ✓  1. IVC DOE R2 (Final).xlsx  (33 rows, 13 cols)
  PASS ✓  2. City of York Council (Final).xlsx  (45 rows, 11 cols)
  PASS ✓  3. Digital Security IT Sample Register (Final).xlsx  (3 rows, 10 cols)
  PASS ✓  4. Moorgate Crossrail Register (Final).xlsx  (10 rows, 10 cols)
  PASS ✓  5. Corporate Risk Register (Final).xlsx  (15 rows, 13 cols)

✅ All files passed. Ready for submission.

Files to include in your submission zip:
  → output/4. Moorgate Crossrail Register (Final).xlsx
  → output/5. Corporate Risk Register (Final).xlsx
  → your code folder with model.py


In [21]:
import os

for _, fname, _, _ in files:
    os.startfile(OUTPUT_DIR / fname)